In [108]:
# Dataframe and Plotting libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn import model_selection
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline


pd.set_option('display.max_columns', None)
# from google.colab import files
import warnings
warnings.filterwarnings('ignore')

In [37]:
# Load the files into a Pandas Dataframe
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
ss = pd.read_csv('SampleSubmission.csv')

In [38]:
print("Train shape: "+ str(train.shape))
print("Test shape: "+ str(test.shape))

Train shape: (373, 12)
Test shape: (558, 11)


In [39]:
train.head()

,MERCHANT_CATEGORIZED_AT,MERCHANT_NAME,MERCHANT_CATEGORIZED_AS,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_AGE,USER_GENDER,USER_HOUSEHOLD,USER_INCOME,USER_ID,Transaction_ID
0,2022-05-04 10:25:50.588042+00,UONSDA CHURCH MOGERE MWAYO TITHE,Rent / Mortgage,1700,2022-05-04 13:56:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_04mk78fa
1,2021-10-25 16:18:38.586837+00,PARK N GO,Transport & Fuel,100,2021-10-24 14:12:00+00,False,25.0,Female,4,90000,ID_U9WZMGJZ,ID_04xkfb07
2,2022-05-20 14:17:30.917297+00,SAFARICOM OFFERS TUNUKIWA,Data & WiFi,20,2022-05-27 12:46:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_051urwuw
3,2022-04-12 15:59:14.139347+00,ZILLIONS CREDIT LIMITED,Loan Repayment,3800,2022-02-07 12:16:00+00,False,NaN,Male,5,252000,ID_3JA0MAFB,ID_0e1om7rz
4,2022-05-04 10:24:26.709045+00,MICHAEL MUSEMBI,Bills & Fees,60,2022-05-04 15:28:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_0kfcoawb


In [40]:
test.head()

,MERCHANT_CATEGORIZED_AT,MERCHANT_NAME,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_AGE,USER_GENDER,USER_HOUSEHOLD,USER_INCOME,USER_ID,Transaction_ID
0,2022-06-01 10:25:16.7131+00,KCB PAYBILL AC,150000,2022-05-05 08:29:00+00,True,NaN,Male,5,150000,ID_O8P8YS18,ID_00x9h2yx
1,2022-03-16 13:05:51.851102+00,IPAY LTD,7394,2019-10-05 16:02:00+00,False,NaN,Female,1,10000,ID_40L9OTIM,ID_01db594f
2,2022-03-10 18:25:42.112294+00,JOHN NJIRU GAKUNJU,200,2021-11-25 09:56:00+00,False,NaN,Female,1,85000,ID_1I8XYBWK,ID_07pm99no
3,2022-04-12 15:59:14.139347+00,UONSDA CHURCH OFFERING-MAIN,50,2022-04-11 14:48:00+00,False,NaN,Male,2,10000,ID_GR569FUO,ID_09jryhkd
4,2022-05-04 10:24:26.709045+00,M-SHWARI LOCK SAVINGS,10000,2022-05-04 13:59:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_0ccv924c


# Prepare

In [41]:
test.isna().sum()

MERCHANT_CATEGORIZED_AT                    0
MERCHANT_NAME                              0
PURCHASE_VALUE                             0
PURCHASED_AT                               0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY      0
USER_AGE                                 473
USER_GENDER                                5
USER_HOUSEHOLD                             0
USER_INCOME                                0
USER_ID                                    0
Transaction_ID                             0
dtype: int64

In [42]:
features = train[['MERCHANT_CATEGORIZED_AT',"PURCHASE_VALUE","PURCHASED_AT","IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY","USER_GENDER","USER_INCOME","USER_HOUSEHOLD"]]

In [43]:
target = train["MERCHANT_CATEGORIZED_AS"]

In [44]:
features["MERCHANT_CATEGORIZED_AT"] = pd.to_datetime(features["MERCHANT_CATEGORIZED_AT"])
features["MERCHANT_CATEGORIZED_AT"] = features['MERCHANT_CATEGORIZED_AT'].dt.hour

In [45]:
features["PURCHASED_AT"] = pd.to_datetime(features["PURCHASED_AT"])
features["PURCHASED_AT"] = features['PURCHASED_AT'].dt.hour

In [46]:
def binate(x):
    if x == True:
        return 1
    elif x == False:
        return 0
    elif x == "Male":
        return 1
    elif x == "Female":
        return 0
    else:
        return 1

In [47]:
features["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"] = features["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"].apply(binate)

In [48]:
features["USER_GENDER"] = features["USER_GENDER"].apply(binate)

In [49]:
features.head()

,MERCHANT_CATEGORIZED_AT,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_GENDER,USER_INCOME,USER_HOUSEHOLD
0,10,1700,13,0,1,10000,3
1,16,100,14,0,0,90000,4
2,14,20,12,0,1,10000,3
3,15,3800,12,0,1,252000,5
4,10,60,15,0,1,10000,3


In [50]:
features.isna().sum()

MERCHANT_CATEGORIZED_AT                  0
PURCHASE_VALUE                           0
PURCHASED_AT                             0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY    0
USER_GENDER                              0
USER_INCOME                              0
USER_HOUSEHOLD                           0
dtype: int64

# Prepare TEST

In [51]:
tfeatures = test[['MERCHANT_CATEGORIZED_AT',"PURCHASE_VALUE","PURCHASED_AT","IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY","USER_GENDER","USER_INCOME","USER_HOUSEHOLD"]]

In [52]:
tfeatures["MERCHANT_CATEGORIZED_AT"] = pd.to_datetime(tfeatures["MERCHANT_CATEGORIZED_AT"])
tfeatures["MERCHANT_CATEGORIZED_AT"] = tfeatures['MERCHANT_CATEGORIZED_AT'].dt.hour

In [53]:
tfeatures["PURCHASED_AT"] = pd.to_datetime(tfeatures["PURCHASED_AT"])
tfeatures["PURCHASED_AT"] = tfeatures['PURCHASED_AT'].dt.hour

In [54]:
tfeatures["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"] = tfeatures["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"].apply(binate)

In [55]:
tfeatures["USER_GENDER"] = tfeatures["USER_GENDER"].apply(binate)

In [56]:
tfeatures.head()

,MERCHANT_CATEGORIZED_AT,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_GENDER,USER_INCOME,USER_HOUSEHOLD
0,10,150000,8,1,1,150000,5
1,13,7394,16,0,0,10000,1
2,18,200,9,0,0,85000,1
3,15,50,14,0,1,10000,2
4,10,10000,13,0,1,10000,3


In [57]:
tfeatures.isna().sum()

MERCHANT_CATEGORIZED_AT                  0
PURCHASE_VALUE                           0
PURCHASED_AT                             0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY    0
USER_GENDER                              0
USER_INCOME                              0
USER_HOUSEHOLD                           0
dtype: int64

In [58]:
tfeatures.shape

(558, 7)

# MODEL

In [165]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import BaggingClassifier, StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier

In [183]:
# ESTIMATORS
svm = LinearSVC()
svm = CalibratedClassifierCV(svm)

tree = DecisionTreeClassifier()

forest = RandomForestClassifier()

light  = LGBMClassifier(learning_rate=0.05)

In [181]:
# ENSEMBLE
bag = BaggingClassifier(base_estimator=CC_svm, n_estimators=10, random_state=0)


decks = [('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
         ('bgc', BaggingClassifier(base_estimator=CC_svm, n_estimators=10, random_state=0))
        ]
stack = StackingClassifier(estimators=decks, final_estimator=LogisticRegression(), stack_method= 'predict_proba')

In [184]:

model = light
model.fit(features, target)

predictions = model.predict_proba(tfeatures)

In [185]:
predictions.shape

(558, 13)

In [186]:
predictions[0:3]

array([[7.69332106e-02, 8.44239322e-03, 5.31333625e-01, 4.28668064e-02,
        2.48438182e-01, 5.94828575e-04, 2.01867466e-03, 7.25076102e-04,
        3.58033461e-04, 1.50955096e-03, 1.32168941e-03, 7.99925723e-02,
        5.46535797e-03],
       [2.54184228e-01, 3.17250927e-03, 7.46256114e-05, 2.93157034e-02,
        2.52514496e-03, 6.50318335e-01, 2.82179016e-02, 2.31043598e-04,
        3.96500886e-04, 1.10493680e-02, 1.00571185e-03, 9.55634709e-03,
        9.95258000e-03],
       [4.85267985e-03, 7.72412385e-04, 2.02529745e-04, 6.85406271e-04,
        3.94820346e-03, 6.43317452e-01, 1.53502011e-01, 1.34585956e-03,
        1.17885582e-04, 1.58511443e-01, 2.20796026e-05, 1.73907497e-02,
        1.53312871e-02]])

In [187]:
model.classes_

array(['Bills & Fees', 'Data & WiFi', 'Education', 'Emergency fund',
       'Family & Friends', 'Going out', 'Groceries', 'Health',
       'Loan Repayment', 'Miscellaneous', 'Rent / Mortgage', 'Shopping',
       'Transport & Fuel'], dtype=object)

In [188]:
predictions_df = pd.DataFrame(predictions)
predictions_df.columns = model.classes_
predictions_df.head()


,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel
0,0.076933,0.008442,0.531334,0.042867,0.248438,0.000595,0.002019,0.000725,0.000358,0.001510,0.001322,0.079993,0.005465
1,0.254184,0.003173,0.000075,0.029316,0.002525,0.650318,0.028218,0.000231,0.000397,0.011049,0.001006,0.009556,0.009953
2,0.004853,0.000772,0.000203,0.000685,0.003948,0.643317,0.153502,0.001346,0.000118,0.158511,0.000022,0.017391,0.015331
3,0.279288,0.084351,0.000155,0.002678,0.021004,0.030372,0.010075,0.000444,0.000885,0.520198,0.000175,0.032139,0.018234
4,0.821055,0.001231,0.000033,0.005134,0.005169,0.011493,0.008939,0.001523,0.006873,0.041365,0.077835,0.008882,0.010469


In [ ]:
def pimp(x):
    if x < 0.025:
        return 0.025
    elif x > .975:
        return .975
    

In [189]:
ss.head()

,Transaction_ID,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel
0,ID_00x9h2yx,0,0,0,0,0,0,0,0,0,0,0,0,0
1,ID_01db594f,0,0,0,0,0,0,0,0,0,0,0,0,0
2,ID_07pm99no,0,0,0,0,0,0,0,0,0,0,0,0,0
3,ID_09jryhkd,0,0,0,0,0,0,0,0,0,0,0,0,0
4,ID_0ccv924c,0,0,0,0,0,0,0,0,0,0,0,0,0


In [190]:
predictions_df['Transaction_ID'] = ss['Transaction_ID']
predictions_df.head()

,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel,Transaction_ID
0,0.076933,0.008442,0.531334,0.042867,0.248438,0.000595,0.002019,0.000725,0.000358,0.001510,0.001322,0.079993,0.005465,ID_00x9h2yx
1,0.254184,0.003173,0.000075,0.029316,0.002525,0.650318,0.028218,0.000231,0.000397,0.011049,0.001006,0.009556,0.009953,ID_01db594f
2,0.004853,0.000772,0.000203,0.000685,0.003948,0.643317,0.153502,0.001346,0.000118,0.158511,0.000022,0.017391,0.015331,ID_07pm99no
3,0.279288,0.084351,0.000155,0.002678,0.021004,0.030372,0.010075,0.000444,0.000885,0.520198,0.000175,0.032139,0.018234,ID_09jryhkd
4,0.821055,0.001231,0.000033,0.005134,0.005169,0.011493,0.008939,0.001523,0.006873,0.041365,0.077835,0.008882,0.010469,ID_0ccv924c


In [191]:
predictions_df = predictions_df[['Transaction_ID', 'Bills & Fees', 'Data & WiFi', 'Education', 'Emergency fund',
       'Family & Friends', 'Going out', 'Groceries', 'Health',
       'Loan Repayment', 'Miscellaneous', 'Rent / Mortgage', 'Shopping',
       'Transport & Fuel']]

In [192]:
# Create the submission csv file
predictions_df.to_csv('lightlr02.csv', index = False)

In [175]:
predictions_df.shape

(558, 14)

In [193]:
model.score(features,target)

0.9624664879356568

In [143]:
log_loss(ss.drop('Transaction_ID',axis=1), predictions_df.drop('Transaction_ID',axis=1))

0.0